In [24]:
import BioSimSpace as bss
from pathlib import Path
import glob
import os
import pandas as pd

# Set repo path

This is only relevant for this example, in your case you will setup the path to your inputs. 

In [25]:
REPO_ROOT = Path().resolve()
DATA_DIR = REPO_ROOT / "data"

In [26]:
ligand_inputs = os.path.join(
    DATA_DIR, "inputs", "ligands"
)

# 1. Load ligand files into BioSimSpace

In this example we'll just be doing this for two ligands by hand. You may want to have a look at [these scripts](https://github.com/OpenBioSim/biosimspace_tutorials/tree/ac200df769ba3373c9e0e724020aaaa96e340889/04_fep/02_RBFE/scripts) to do the setup in a batch.

In [27]:
ligand_1_file = os.path.join(ligand_inputs, "ligand_1.sdf")

ligand_1_directory = os.path.join(ligand_inputs, "ligand_1")

# This will raise an error if the directory already exists, to avoid overwriting files. You can set exist_ok=True:
os.makedirs(ligand_1_directory, exist_ok=False) 

ligand_1_molecule = bss.IO.readMolecules(ligand_1_file)[0]

In [28]:
ligand_2_file = os.path.join(ligand_inputs, "ligand_2.sdf")

ligand_2_directory = os.path.join(ligand_inputs, "ligand_2")

# This will raise an error if the directory already exists, to avoid overwriting files. You can set exist_ok=True:
os.makedirs(ligand_2_directory, exist_ok=False) 

ligand_2_molecule = bss.IO.readMolecules(ligand_2_file)[0]

# 2. Parameterise ligands

In [29]:
parameterised_ligand_1 = bss.Parameters.gaff2(
    molecule=ligand_1_molecule,
    net_charge=-1,
    work_dir=ligand_1_directory
).getMolecule()

In [30]:
parameterised_ligand_2 = bss.Parameters.gaff2(
    molecule=ligand_2_molecule,
    net_charge=-1,
    work_dir=ligand_2_directory
).getMolecule()

# 3. Create box

In [31]:
box_axis_length = 12.0 #nanometers

box_1_min, box_1_max = parameterised_ligand_1.getAxisAlignedBoundingBox()
box_2_min, box_2_max = parameterised_ligand_2.getAxisAlignedBoundingBox()


In [32]:
box_1_size = [y - x for x, y in zip(box_1_min, box_1_max)]
box_2_size = [y - x for x, y in zip(box_2_min, box_2_max)]
box_1_sizes = [x + int(box_axis_length) * bss.Units.Length.nanometer for x in box_1_size]
box_2_sizes = [x + int(box_axis_length) * bss.Units.Length.nanometer for x in box_2_size]


In [33]:
box_1, angles_1 = bss.Box.cubic(max(box_1_sizes))
box_2, angles_2 = bss.Box.cubic(max(box_2_sizes))

# 4. Solvate

In [34]:
ligand_1_solvated = bss.Solvent.solvate(
    model="tip3p", 
    molecule=parameterised_ligand_1, 
    box=box_1, 
    angles=angles_1, 
    ion_conc=0.15,
    work_dir=ligand_1_directory
)

In [35]:
ligand_2_solvated = bss.Solvent.solvate(
    model="tip3p", 
    molecule=parameterised_ligand_2, 
    box=box_2, 
    angles=angles_2, 
    ion_conc=0.15,
    work_dir=ligand_2_directory
)

# 5. Save solvated systems

In [36]:
ligand_1_name = os.path.join(ligand_1_directory, "solvated_ligand_1_unbound")
ligand_2_name = os.path.join(ligand_2_directory, "solvated_ligand_2_unbound")

bss.IO.saveMolecules(
    filebase=ligand_1_name,
    system=ligand_1_solvated,
    fileformat=["prm7", "rst7"]
)


bss.IO.saveMolecules(
    filebase=ligand_2_name,
    system=ligand_2_solvated,
    fileformat=["prm7", "rst7"]
)

['/Users/af25016/projects/ligand_rbfe/data/inputs/ligands/ligand_2/solvated_ligand_2_unbound.prm7',
 '/Users/af25016/projects/ligand_rbfe/data/inputs/ligands/ligand_2/solvated_ligand_2_unbound.rst7']